*Imports*

In [1]:
import sys
import os
sys.path.append("/media/mrsmile/IA/tesis/")


In [2]:
import json
import torch
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from datetime import datetime
from tqdm import tqdm
import matplotlib.pyplot as plt
import numpy as np
from models.resunet_encoder import ResUNetEncoder
from models.autoencoder_pretrain import ResUNetAutoEncoder
from src.dataset_patches_memmap import CTAMemmapDataset
torch.backends.cudnn.benchmark = True




In [3]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
EPOCHS = 250
BATCH_SIZE = 5
LR_MAX = 1e-4  
LR_MIN = 1e-6  

PATCH_SIZE = (128,128,128)
PATCHES_PER_VOLUME = 10

ROOT_MEMMAP = "/media/mrsmile/IA/tesis/data/processed/memmap"
JSON_PATH = "/media/mrsmile/IA/tesis/data/metadata/data_split.json"
LOG_DIR = "/media/mrsmile/IA/tesis/runs/tensorboard/pretrain"
CKPT_DIR = "/media/mrsmile/IA/tesis/runs/checkpoints/pretraining"

os.makedirs(LOG_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)



In [4]:
print(DEVICE)

cuda


In [5]:
def get_file_lists(json_path):
    with open(json_path, 'r') as f:
        data = json.load(f)
    train_list = [f.replace(".nii.gz", "") for f in data['imagecas']['train']]
    val_list = [f.replace(".nii.gz", "") for f in data['imagecas']['val']]
    return train_list, val_list

In [6]:
train_files, val_files = get_file_lists(JSON_PATH)

In [8]:
def seed_worker(worker_id):
    np.random.seed(torch.initial_seed() % 2**32 + worker_id)

In [9]:
train_dataset = CTAMemmapDataset(
    root=ROOT_MEMMAP,
    patch_size=PATCH_SIZE,
    patches_per_volume=PATCHES_PER_VOLUME,
    file_list=train_files
)

val_dataset = CTAMemmapDataset(
    root=ROOT_MEMMAP,
    patch_size=PATCH_SIZE,
    patches_per_volume=PATCHES_PER_VOLUME,
    file_list=val_files
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    prefetch_factor=2,
    persistent_workers=True,
    worker_init_fn=seed_worker  
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    prefetch_factor=1,
    persistent_workers=True
)

In [10]:
# --- MODELO Y OPTIMIZACIÓN ---
model = ResUNetAutoEncoder().to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR_MAX, weight_decay=1e-5)
criterion = torch.nn.L1Loss()
scaler = torch.amp.GradScaler('cuda')
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS, eta_min=LR_MIN
)
print(f"Modelo en: {DEVICE}")
print(f"Parámetros totales: {sum(p.numel() for p in model.parameters()):,}")

Modelo en: cuda
Parámetros totales: 3,900,961


In [11]:
# --- LOGGING ---
run_name = datetime.now().strftime("%Y%m%d_%H%M")
log_path = os.path.join(LOG_DIR, "Pretraining", f"pretrain_final_{run_name}")
writer = SummaryWriter(log_dir=log_path)
current_run_ckpt_dir = os.path.join(CKPT_DIR, f"run_{run_name}")
os.makedirs(current_run_ckpt_dir, exist_ok=True)

print(f"Checkpoints en: {current_run_ckpt_dir}")
print(f"TensorBoard en: {log_path}")

Checkpoints en: /media/mrsmile/IA/tesis/runs/checkpoints/pretraining/run_20260305_0045
TensorBoard en: /media/mrsmile/IA/tesis/runs/tensorboard/pretrain/Pretraining/pretrain_final_20260305_0045


In [12]:
# --- LOOP DE ENTRENAMIENTO ---
global_step = 0
best_val_loss = float('inf')

for epoch in range(EPOCHS):

    # TRAIN
    model.train()
    epoch_train_loss = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS} [TRAIN]")
    for x in pbar:
        x = x.to(DEVICE, non_blocking=True)
        x_noisy = x + 0.05 * torch.randn_like(x)

        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            out = model(x_noisy)
            loss = criterion(out, x)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        epoch_train_loss += loss.item()
        writer.add_scalar("Loss/batch", loss.item(), global_step)
        global_step += 1
        pbar.set_postfix(
            loss=f"{loss.item():.4f}",
            lr=f"{optimizer.param_groups[0]['lr']:.2e}"
        )

    # VALIDACIÓN
    model.eval()
    epoch_val_loss = 0
    with torch.no_grad():
        for x_val in val_loader:
            x_val = x_val.to(DEVICE, non_blocking=True)
            with torch.amp.autocast('cuda'):
                val_out = model(x_val)
                v_loss = criterion(val_out, x_val)
            epoch_val_loss += v_loss.item()

    # SCHEDULER
    scheduler.step()

    # PROMEDIOS
    avg_train = epoch_train_loss / len(train_loader)
    avg_val   = epoch_val_loss   / len(val_loader)
    current_lr = optimizer.param_groups[0]['lr']

    # TENSORBOARD
    writer.add_scalar("Loss/train_epoch", avg_train, epoch)
    writer.add_scalar("Loss/val_epoch",   avg_val,   epoch)
    writer.add_scalar("Loss/train_val_gap", avg_train - avg_val, epoch)
    writer.add_scalar("LR", current_lr, epoch)

    # VISUALIZACIÓN DE RECONSTRUCCIÓN cada 10 épocas
    if epoch % 10 == 0:
        model.eval()
        with torch.no_grad():
            x_sample = next(iter(val_loader)).to(DEVICE)
            x_noisy_vis = x_sample + 0.05 * torch.randn_like(x_sample)
            x_recon = model(x_noisy_vis)
            slice_idx = x_sample.shape[-1] // 2
            writer.add_image("Reconstruccion/original",
                x_sample[0, 0, :, :, slice_idx].unsqueeze(0), epoch)
            writer.add_image("Reconstruccion/ruidoso",
                x_noisy_vis[0, 0, :, :, slice_idx].unsqueeze(0), epoch)
            writer.add_image("Reconstruccion/reconstruido",
                x_recon[0, 0, :, :, slice_idx].unsqueeze(0), epoch)

    print(f"Epoch {epoch:03d} | train={avg_train:.4f} | val={avg_val:.4f} | lr={current_lr:.2e}")

    # GUARDAR MEJOR MODELO
    if avg_val < best_val_loss:
        best_val_loss = avg_val
        best_path = os.path.join(current_run_ckpt_dir, "encoder_best.pth")
        torch.save(model.enc.state_dict(), best_path)
        print(f"Mejor encoder guardado (val={best_val_loss:.4f}) en {best_path}")

    # CHECKPOINT CADA ÉPOCA
    epoch_path = os.path.join(current_run_ckpt_dir, f"encoder_epoch_{epoch}.pth")
    torch.save(model.enc.state_dict(), epoch_path)
    print(f"Checkpoint guardado: {epoch_path}")

writer.close()
print(f"\nEntrenamiento completo. Mejor val_loss: {best_val_loss:.4f}")

Epoch 0/250 [TRAIN]: 100%|██████████| 1400/1400 [04:01<00:00,  5.80it/s, loss=0.1121, lr=1.00e-04]


Epoch 000 | train=0.2004 | val=0.1608 | lr=1.00e-04
Mejor encoder guardado (val=0.1608) en /media/mrsmile/IA/tesis/runs/checkpoints/pretraining/run_20260305_0045/encoder_best.pth
Checkpoint guardado: /media/mrsmile/IA/tesis/runs/checkpoints/pretraining/run_20260305_0045/encoder_epoch_0.pth


Epoch 1/250 [TRAIN]: 100%|██████████| 1400/1400 [03:56<00:00,  5.91it/s, loss=0.1002, lr=1.00e-04]


Epoch 001 | train=0.1263 | val=0.1337 | lr=1.00e-04
Mejor encoder guardado (val=0.1337) en /media/mrsmile/IA/tesis/runs/checkpoints/pretraining/run_20260305_0045/encoder_best.pth
Checkpoint guardado: /media/mrsmile/IA/tesis/runs/checkpoints/pretraining/run_20260305_0045/encoder_epoch_1.pth


Epoch 2/250 [TRAIN]: 100%|██████████| 1400/1400 [03:53<00:00,  5.99it/s, loss=0.1088, lr=1.00e-04]


...

Epoch 155 | train=0.0501 | val=0.0531 | lr=3.17e-05
Checkpoint guardado: /media/mrsmile/IA/tesis/runs/checkpoints/pretraining/run_20260305_0045/encoder_epoch_155.pth


Epoch 156/250 [TRAIN]: 100%|██████████| 1400/1400 [03:57<00:00,  5.90it/s, loss=0.0400, lr=3.17e-05]


Epoch 156 | train=0.0502 | val=0.0529 | lr=3.11e-05
Mejor encoder guardado (val=0.0529) en /media/mrsmile/IA/tesis/runs/checkpoints/pretraining/run_20260305_0045/encoder_best.pth
Checkpoint guardado: /media/mrsmile/IA/tesis/runs/checkpoints/pretraining/run_20260305_0045/encoder_epoch_156.pth


Epoch 157/250 [TRAIN]:   7%|▋         | 98/1400 [00:17<03:47,  5.73it/s, loss=0.0427, lr=3.11e-05]


KeyboardInterrupt: 

In [ ]:
enc = ResUNetEncoder().cuda()
state = torch.load("/media/mrsmile/IA/tesis/runs/checkpoints/pretraining/encoder_epoch_80.pth")
enc.load_state_dict(state)
enc.eval()

In [ ]:
x = next(iter(loader)).cuda()  # (1,1,112,112,112)


In [ ]:
with torch.no_grad():
    feat = enc(x)   # (1,C,D,H,W)


In [ ]:
import matplotlib.pyplot as plt

x_np = x[0,0].cpu().numpy()      # (112,112,112)
f_np = feat[0,0].cpu().numpy()   # (14,14,14)

Dx = x_np.shape[0]
Df = f_np.shape[0]

# slices independientes
x_slices = [Dx//2 - 20, Dx//2 - 10, Dx//2, Dx//2 + 10]
f_slices = [Df//2 - 2, Df//2 - 1, Df//2, Df//2 + 1]

fig, axs = plt.subplots(2, 4, figsize=(16,8))

for i in range(4):
    axs[0,i].imshow(x_np[x_slices[i]], cmap="gray")
    axs[0,i].set_title(f"Input {x_slices[i]}")
    axs[0,i].axis("off")

    axs[1,i].imshow(f_np[f_slices[i]], cmap="hot")
    axs[1,i].set_title(f"Encoder {f_slices[i]}")
    axs[1,i].axis("off")

plt.suptitle("Input vs Encoder activations", fontsize=16)
plt.show()


In [ ]:
import napari
import numpy as np

# input
x_np = x[0, 0].cpu().numpy()        # (112,112,112)

# encoder activación (baja resolución)
f_np = feat[0, 0].cpu().numpy()     # (14,14,14)

# normalizamos para ver mejor
x_np = (x_np - x_np.min()) / (x_np.max() - x_np.min())
f_np = (f_np - f_np.min()) / (f_np.max() - f_np.min())

viewer = napari.Viewer()

viewer.add_image(
    x_np,
    name="CTA input",
    colormap="gray",
    contrast_limits=(0, 1)
)

viewer.add_image(
    f_np,
    name="Encoder activation",
    colormap="hot",
    opacity=0.6
)

napari.run()


In [ ]:
plt.plot(f_np.mean(axis=(1,2)))


In [ ]:
x1 = next(iter(loader))
x2 = next(iter(loader))


In [ ]:
x1_np = x1[0,0].cpu().numpy()
mid = x1_np.shape[0] // 2

plt.imshow(x1_np[mid], cmap="gray")
plt.title("Slice input CTA (x1)")
plt.colorbar()
plt.show()
with torch.no_grad():
    f1 = enc(x1.cuda())
    f2 = enc(x2.cuda())
print(f1.mean().item(), f2.mean().item())
 

In [ ]:
fig, axs = plt.subplots(4, 4, figsize=(12,12))
for i in range(16):
    axs[i//4, i%4].imshow(feat[0,i,7].cpu(), cmap="hot")
    axs[i//4, i%4].set_title(f"Ch {i}")
    axs[i//4, i%4].axis("off")
plt.show()


In [ ]:
x_noisy = x + 0.1 * torch.randn_like(x)
feat_noisy = enc(x_noisy)


In [ ]:
from sklearn.decomposition import PCA
X = feat.reshape(-1, feat.shape[1]).cpu().numpy()
X2 = PCA(2).fit_transform(X)
